# 02 Bronze

Auto Loader lee `landing/` con el esquema explícito de la Parte 1 y hace append
a la tabla. Bronze guarda el mensaje tal como llegó. El checkpoint ya garantiza
que cada archivo se lee una vez. La deduplicación por `(MMSI, BaseDateTime)`
ocurre en silver. `fecha` es el widget del job y solo etiqueta la métrica.

No ejecutes este stream en dos sesiones a la vez porque el checkpoint es uno solo.
`useIncrementalListing=false` lista la carpeta completa. Con un CSV por día es rapido, y un archivo nuevo no depende de quedar delante del último nombre ya visto.


In [ ]:
# Capturar los parámetros del notebook llamador antes de %run.
dbutils.widgets.text("fecha", "2023-06-08")
dbutils.widgets.text("ancla_replay", "")
dbutils.widgets.text("inicio_run", "")
ow_argumentos = tuple(dbutils.widgets.get(k).strip() for k in ("fecha", "ancla_replay", "inicio_run"))

In [ ]:
%run ./_config

In [ ]:
from pyspark.sql import functions as F

ais_schema = """
    MMSI string,
    BaseDateTime timestamp,
    LAT double,
    LON double,
    SOG double,
    COG double,
    Heading double,
    VesselName string,
    IMO string,
    CallSign string,
    VesselType int,
    Status int,
    Length double,
    Width double,
    Draft double,
    Cargo string,
    TransceiverClass string
"""


def leer_landing(landing=LANDING, esquema=SCHEMA_LOC):
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", esquema)
        .option("cloudFiles.schemaEvolutionMode", "none")
        .option("cloudFiles.includeExistingFiles", "true")
        .option("cloudFiles.allowOverwrites", "false")
        .option("cloudFiles.useIncrementalListing", "false")
        .option("header", "true")
        .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
        .schema(ais_schema)
        .load(landing)
        .select(
            "MMSI", "BaseDateTime", "LAT", "LON", "SOG", "COG", "Heading",
            "VesselName", "IMO", "CallSign", "VesselType", "Status",
            "Length", "Width", "Draft", "Cargo", "TransceiverClass",
            F.col("_metadata.file_path").alias("_source_file"),
            F.current_timestamp().alias("_ingested_at"),
        )
    )


def filas(tabla: str) -> int:
    return int(spark.sql(f"SELECT COUNT(*) AS n FROM {tabla}").collect()[0]["n"])

## Ingesta del job o ensayo aislado

La opción normal consume el landing del equipo. El ensayo verifica el mismo archivo, el mismo checkpoint y un reenvío con nombre nuevo.

In [ ]:
dbutils.widgets.dropdown("prueba_bronze","no",["no","si"])
if dbutils.widgets.get("prueba_bronze")=="si":
    # Ensayo aislado con el mismo lector cloudFiles de producción.
    from pathlib import Path
    import uuid
    clave = "ow_bronze_prueba_" + uuid.uuid4().hex[:10]
    base = Path(VOLUME_ROOT)/"ensayos"/clave
    (base/"landing").mkdir(parents=True)
    cabecera = "MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass"
    mensaje = "123456789,2023-06-08T12:00:00,28,-85,5,40,,,,,,,,,,,A"
    def llegada(nombre):
        (base/"landing"/nombre).write_text(cabecera+"\n"+mensaje+"\n",encoding="utf-8")
    tabla_demo = f"`{CATALOG}`.`{SCHEMA}`.`{clave}`"
    def leer_prueba():
        q = (leer_landing(str(base/"landing"),str(base/"esquema")).writeStream.format("delta")
             .outputMode("append").option("checkpointLocation",str(base/"checkpoint"))
             .trigger(availableNow=True).toTable(tabla_demo))
        q.awaitTermination()
        return spark.table(tabla_demo).count()
    llegada("a.csv")
    primero, repetido = leer_prueba(), leer_prueba()
    llegada("b.csv")
    nuevo = leer_prueba()
    assert (primero,repetido,nuevo)==(1,1,2), (primero,repetido,nuevo)
    print("Archivos en Bronze · inicial, mismo checkpoint, reenvío con otro nombre:",primero,repetido,nuevo,tabla_demo)
else:
    for consulta in list(spark.streams.active):
        if consulta.name == "oceanwatch_bronze_ais":
            consulta.stop()

    antes = filas(BRONZE)
    consulta = (
        leer_landing()
        .writeStream.format("delta")
        .outputMode("append")
        .queryName("oceanwatch_bronze_ais")
        .option("checkpointLocation", CHECKPOINT_BRONZE)
        .trigger(availableNow=True)
        .toTable(BRONZE)
    )
    consulta.awaitTermination()
    despues = filas(BRONZE)

    spark.sql(f"""
    INSERT INTO {METRICAS}
    SELECT
      DATE '{fecha_dia.isoformat()}',
      'bronze',
      CAST({antes} AS BIGINT),
      CAST({despues} AS BIGINT),
      CAST(NULL AS BIGINT),
      CAST(NULL AS BIGINT),
      CAST(NULL AS BIGINT),
      current_timestamp()
    """)
    print(f"Ejecución bronze {fecha_dia.isoformat()}: antes={antes} despues={despues}")